## Proyecto Integrador – Fase 1: Modelo Predictivo de Abandono de Clientes

**Asignatura:** Modelos y Simulación de Sistemas I

**Integrantes:** Shara Olaya Araque, Liceth Alexandra Sarmiento

## 1. Introducción

### Problema
Las empresas de telecomunicaciones pierden ingresos cuando sus clientes
cancelan el servicio. Identificar con anticipación qué clientes tienen mayor
probabilidad de abandonar permite a la empresa tomar acciones de retención
(ofertas, mejoras en el servicio, contacto personalizado) antes de que el
abandono ocurra.

### Objetivo
Construir un modelo de Machine Learning que, a partir de las características
de un cliente (servicios contratados, tipo de contrato, cargos, antigüedad,
entre otras), prediga si el cliente abandonará el servicio.

### Tipo de problema
Clasificación binaria supervisada: el modelo aprende a partir de ejemplos
etiquetados y debe asignar a cada cliente una de dos clases.

### Variable objetivo
`Churn`: indica si el cliente abandonó el servicio (`Yes`) o permaneció (`No`).

### Fuente de datos
Dataset *Telco Customer Churn*, disponible en Kaggle:
https://www.kaggle.com/datasets/blastchar/telco-customer-churn

**Configuración del entorno**

En esta sección se importan las librerías base del proyecto y se registran sus versiones, con el fin de garantizar la reproducibilidad. Además, se define la constante RANDOM_STATE = 42, que se utilizará en todos los procesos que involucren aleatoriedad (división de datos, entrenamiento de modelos), de modo que los resultados sean los mismos en cada ejecución.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import sklearn
import joblib

# Semilla aleatoria fija para todo el proyecto
RANDOM_STATE = 42

print("Versión de Python:", sys.version)
print("Versión de numpy:", np.__version__)
print("Versión de pandas:", pd.__version__)
print("Versión de matplotlib:", matplotlib.__version__)
print("Versión de scikit-learn:", sklearn.__version__)
print("Versión de joblib:", joblib.__version__)

Versión de Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Versión de numpy: 2.1.3
Versión de pandas: 2.2.3
Versión de matplotlib: 3.10.0
Versión de scikit-learn: 1.6.1
Versión de joblib: 1.6.0


**Carga del dataset;** El archivo CSV se encuentra almacenado en el repositorio del proyecto y se carga directamente desde GitHub, de modo que el notebook pueda ejecutarse sin descargas ni cargas manuales de archivos.

In [2]:
# Carga del dataset
# URL "raw" del CSV guardado en el repositorio de GitHub
URL_DATASET = "https://raw.githubusercontent.com/Alexa0723/Proyecto_Modelos1/refs/heads/main/fase-1/data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Leer el CSV y convertirlo en un DataFrame
datos_clientes = pd.read_csv(URL_DATASET)

# Dimensiones del dataset
numero_de_registros = datos_clientes.shape[0]
numero_de_variables = datos_clientes.shape[1]

print("Número de registros (filas):", numero_de_registros)
print("Número de variables (columnas):", numero_de_variables)

Número de registros (filas): 7043
Número de variables (columnas): 21


In [3]:
# Mostrar las primeras 5 filas del dataset
datos_clientes.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# Mostrar la lista completa de nombres de columnas
print(datos_clientes.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


## 2. Descripción del dataset

El dataset contiene 7043 registros,donde cada registro corresponde a un
cliente, y 21 variables, que pueden agruparse así:

- **Identificador:** customerID, código único de cada cliente.
- **Datos demográficos:** gender, SeniorCitizen, Partner, `Dependents`.
- **Servicios contratados:** `PhoneService`, `MultipleLines`, `InternetService`,
  `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`,
  `StreamingTV`, `StreamingMovies`.
- **Cuenta y facturación:** `tenure` (meses como cliente), `Contract`,
  `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`.
- **Variable objetivo:** `Churn`, que indica si el cliente abandonó el
  servicio (`Yes`) o permaneció (`No`).

Observaciones iniciales:
- `customerID` es un identificador, por lo que no aporta información
  predictiva; su exclusión se justifica más adelante.
- `SeniorCitizen` está codificada como 0/1, aunque representa una variable
  categórica (si el cliente es adulto mayor o no).
- Algunas variables de servicios contienen categorías como
  "No phone service", que no corresponden a valores faltantes sino a
  clientes que no tienen el servicio base.

**Limitaciones del dataset:**
- Los datos corresponden a un único momento en el tiempo; no incluyen fechas,
  por lo que no es posible saber cuándo abandonó cada cliente ni analizar la
  evolución de su comportamiento.
- La variable objetivo está desbalanceada (26,54 % de abandono), lo que
  condiciona la elección de métricas.
- `TotalCharges` presenta 11 valores faltantes, correspondientes a clientes
  nuevos.

## 3. Exploración de datos

### 3.1 Tipos de datos
Se revisa el tipo de dato con el que pandas almacenó cada variable, para
verificar que coincida con lo que la variable representa.

In [5]:
# 3.1 Tipos de datos

# Resumen de columnas: tipo de dato y cantidad de valores no nulos
datos_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


El dataset contiene 18 variables de tipo texto (object), 2 de tipo entero
(int64) y 1 de tipo decimal (float64). Según `.info()`, todas las columnas
tienen 7043 valores no nulos.

Se identifican dos situaciones que requieren atención:

- `TotalCharges` fue almacenada como texto (object), aunque representa un
  valor monetario. Esto indica que la columna contiene al menos un valor que
  pandas no pudo interpretar como número, por lo que se analiza en la
  sección 3.3.
- `SeniorCitizen` es de tipo int64, pero representa una variable categórica
  binaria (0 = no es adulto mayor, 1 = sí lo es), por lo que no debe
  tratarse como una cantidad.

### 3.2 Verificación del identificador
Se comprueba si `customerID` identifica de forma única a cada cliente.

In [6]:
# 3.2 Verificación del identificador customerID

cantidad_ids_unicos = datos_clientes["customerID"].nunique()
cantidad_total_registros = len(datos_clientes)

print("Valores únicos en customerID:", cantidad_ids_unicos)
print("Total de registros:", cantidad_total_registros)

if cantidad_ids_unicos == cantidad_total_registros:
    print("Cada cliente tiene un customerID distinto: la columna es un identificador único.")
else:
    print("Hay valores de customerID repetidos: es necesario revisarlos.")

Valores únicos en customerID: 7043
Total de registros: 7043
Cada cliente tiene un customerID distinto: la columna es un identificador único.


`customerID` tiene 7043 valores únicos para 7043 registros. Esto confirma
que cada cliente aparece una sola vez (no hay registros duplicados) y que la
columna funciona como identificador. Como no describe ninguna característica
del cliente, no se utilizará como variable predictora: si el modelo la usara,
podría memorizar clientes específicos en lugar de aprender patrones
generalizables.

### 3.3 Valores faltantes
Se cuantifican los valores faltantes en dos pasos: primero los valores nulos
(NaN) que pandas detecta directamente, y luego los textos vacíos, que pandas
no reconoce como faltantes.

In [7]:
# 3.3 Valores faltantes (NaN)

faltantes_por_columna = datos_clientes.isnull().sum()

print("Valores faltantes (NaN) por columna:")
print(faltantes_por_columna)
print()
print("Total de valores faltantes (NaN) en el dataset:", faltantes_por_columna.sum())

Valores faltantes (NaN) por columna:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Total de valores faltantes (NaN) en el dataset: 0


La función `isnull()` no detecta valores nulos (NaN) en ninguna columna.
Sin embargo, esto no garantiza la ausencia de datos faltantes: un texto vacío
o compuesto solo por espacios no es reconocido por pandas como valor nulo.
Por esta razón, se realiza una búsqueda adicional en las columnas de texto.

### 3.4 Estadísticas descriptivas

In [8]:
# 3.4 Búsqueda de valores vacíos escondidos en columnas de texto

columnas_de_texto = datos_clientes.select_dtypes(include="object").columns
se_encontraron_vacios = False

for nombre_columna in columnas_de_texto:
    cantidad_vacios = (datos_clientes[nombre_columna].str.strip() == "").sum()

    if cantidad_vacios > 0:
        print("Columna", nombre_columna, "tiene", cantidad_vacios, "valores vacíos")
        se_encontraron_vacios = True

if se_encontraron_vacios == False:
    print("No se encontraron valores vacíos escondidos en las columnas de texto.")

Columna TotalCharges tiene 11 valores vacíos


Se encontraron 11 valores vacíos en la columna TotalCharges. Estos valores
explican por qué pandas almacenó la columna como texto. Ninguna otra columna
de texto presenta valores vacíos.

In [9]:
# 3.5 Conversión de TotalCharges a tipo numérico

print("Tipo de TotalCharges antes de la conversión:", datos_clientes["TotalCharges"].dtype)

datos_clientes["TotalCharges"] = pd.to_numeric(datos_clientes["TotalCharges"], errors="coerce")

print("Tipo de TotalCharges después de la conversión:", datos_clientes["TotalCharges"].dtype)
print("Valores faltantes (NaN) en TotalCharges después de la conversión:",
      datos_clientes["TotalCharges"].isnull().sum())

Tipo de TotalCharges antes de la conversión: object
Tipo de TotalCharges después de la conversión: float64
Valores faltantes (NaN) en TotalCharges después de la conversión: 11


`TotalCharges` se convirtió de texto (object) a número decimal (float64)
mediante `pd.to_numeric` con `errors="coerce"`, que transforma los valores no
convertibles en NaN. Después de la conversión, la columna tiene exactamente
11 valores NaN, lo que coincide con los 11 vacíos encontrados. Esto confirma
que no existían otros valores no numéricos en la columna.

Esta conversión no genera fuga de información, ya que se aplica de forma
independiente a cada fila y no utiliza información de otros registros.

In [10]:
# 3.6 Análisis de las filas con TotalCharges faltante

filas_sin_total_charges = datos_clientes[datos_clientes["TotalCharges"].isnull()]

cantidad_filas_sin_total = len(filas_sin_total_charges)
porcentaje_filas_sin_total = cantidad_filas_sin_total / len(datos_clientes) * 100

print("Filas con TotalCharges faltante:", cantidad_filas_sin_total)
print("Porcentaje del total de registros:", round(porcentaje_filas_sin_total, 2), "%")

filas_sin_total_charges[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

Filas con TotalCharges faltante: 11
Porcentaje del total de registros: 0.16 %


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,NaN,Two year,No
753,3115-CZMZD,0,20.25,NaN,Two year,No
936,5709-LVOEQ,0,80.85,NaN,Two year,No
1082,4367-NUYAO,0,25.75,NaN,Two year,No
1340,1371-DWPAZ,0,56.05,NaN,Two year,No
3331,7644-OMVMY,0,19.85,NaN,Two year,No
3826,3213-VVOLG,0,25.35,NaN,Two year,No
4380,2520-SGTTA,0,20.00,NaN,Two year,No
5218,2923-ARZLG,0,19.70,NaN,One year,No
6670,4075-WKNIU,0,73.35,NaN,Two year,No


In [14]:
# 3.7 ¿Los faltantes corresponden a clientes con tenure = 0?

cantidad_clientes_tenure_cero = (datos_clientes["tenure"] == 0).sum()
print("Clientes con tenure = 0:", cantidad_clientes_tenure_cero)

faltantes_con_tenure_cero = (filas_sin_total_charges["tenure"] == 0).sum()
print("Filas con TotalCharges faltante que tienen tenure = 0:", faltantes_con_tenure_cero)

if faltantes_con_tenure_cero == cantidad_filas_sin_total:
    print("Todas las filas con TotalCharges faltante son clientes con tenure = 0.")
else:
    print("No todas las filas con TotalCharges faltante tienen tenure = 0: hay que revisar.")

Clientes con tenure = 0: 11
Filas con TotalCharges faltante que tienen tenure = 0: 11
Todas las filas con TotalCharges faltante son clientes con tenure = 0.


Los 11 registros con `TotalCharges` faltante representan el 0,16 % del
dataset. Al analizarlos se observa que:

- Todos tienen `tenure = 0`, es decir, son clientes que acaban de ingresar a
  la empresa. Además, son los únicos 11 clientes del dataset con `tenure = 0`.
- Todos tienen un valor en `MonthlyCharges`, es decir, ya tienen una tarifa
  asignada.
- Todos tienen `Churn = No`.
- 10 tienen contrato `Two year` y 1 tiene contrato `One year`.

Por lo tanto, los valores faltantes no son aleatorios ni corresponden a
errores de captura: se trata de clientes nuevos a los que todavía no se les ha
realizado ningún cobro, por lo que su cargo total acumulado aún no existe.

El tratamiento de estos valores se decide en la sección 4. Cualquier
imputación se ajustará únicamente con los datos de entrenamiento, para evitar
fuga de información.

In [15]:
# 3.8 Estadísticas descriptivas: variables numéricas

datos_clientes.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7032.000000
mean,0.162147,32.371149,64.761692,2283.300441
std,0.368612,24.559481,30.090047,2266.771362
min,0.000000,0.000000,18.250000,18.800000
25%,0.000000,9.000000,35.500000,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.850000,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


Observaciones sobre las variables numéricas:

- **SeniorCitizen:** al ser una variable binaria (0/1), su media corresponde a
  la proporción de clientes adultos mayores: 16,2 %.
- **tenure:** varía entre 0 y 72 meses, con media de 32,37 y mediana de 29
  meses.
- **MonthlyCharges:** varía entre 18,25 y 118,75, con media de 64,76 y mediana
  de 70,35. La mediana es mayor que la media, lo que indica un grupo de
  clientes con cargos mensuales bajos.
- **TotalCharges:** tiene 7032 valores (7043 menos los 11 faltantes). Varía
  entre 18,80 y 8684,80, con media de 2283,30 y mediana de 1397,48. La media
  es considerablemente mayor que la mediana, lo que indica una distribución
  sesgada a la derecha: pocos clientes con cargos acumulados muy altos.

No se observan valores negativos ni fuera de rangos razonables. Las variables
numéricas se encuentran en escalas muy diferentes (por ejemplo, `tenure`
llega hasta 72, mientras que `TotalCharges` llega hasta 8684,80), lo cual
se considerará en la preparación de los datos.

In [16]:
# 3.9 Estadísticas descriptivas: variables categóricas

datos_clientes.describe(include="object")

,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,7043,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,2
top,3186-AJIEK,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,No
freq,1,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,5174


Observaciones sobre las variables categóricas:

- La mayoría de variables tiene 2 o 3 categorías; `PaymentMethod` tiene 4.
- `gender` está prácticamente balanceada (Male: 3555 clientes, 50,48 %).
- El 90,32 % de los clientes tiene servicio telefónico (`PhoneService = Yes`).
- El tipo de contrato más frecuente es `Month-to-month` (3875 clientes,
  55,02 %).
- El método de pago más frecuente es `Electronic check` (2365 clientes,
  33,58 %).
- **Variable objetivo:** 5174 clientes (73,46 %) tienen `Churn = No`, por lo
  que 1869 clientes (26,54 %) abandonaron el servicio. Las clases están
  desbalanceadas; esto se analiza en la sección 3.5 y se tiene en cuenta en
  la elección de las métricas de evaluación.

In [17]:
# 3.10 Categorías de cada variable categórica

columnas_categoricas = datos_clientes.select_dtypes(include="object").columns

for nombre_columna in columnas_categoricas:
    if nombre_columna == "customerID":
        continue
    else:
        categorias = datos_clientes[nombre_columna].unique()
        print(nombre_columna, ":", list(categorias))

gender : ['Female', 'Male']
Partner : ['Yes', 'No']
Dependents : ['No', 'Yes']
PhoneService : ['No', 'Yes']
MultipleLines : ['No phone service', 'No', 'Yes']
InternetService : ['DSL', 'Fiber optic', 'No']
OnlineSecurity : ['No', 'Yes', 'No internet service']
OnlineBackup : ['Yes', 'No', 'No internet service']
DeviceProtection : ['No', 'Yes', 'No internet service']
TechSupport : ['No', 'Yes', 'No internet service']
StreamingTV : ['No', 'Yes', 'No internet service']
StreamingMovies : ['No', 'Yes', 'No internet service']
Contract : ['Month-to-month', 'One year', 'Two year']
PaperlessBilling : ['Yes', 'No']
PaymentMethod : ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']
Churn : ['No', 'Yes']


Las categorías de cada variable no presentan errores de escritura ni valores
inesperados. `Churn` tiene exactamente dos valores (`Yes` y `No`).

Se identifican categorías que dependen de otra variable:

- `"No phone service"` en `MultipleLines` corresponde a clientes con
  `PhoneService = No`.
- `"No internet service"` aparece en `OnlineSecurity`, `OnlineBackup`,
  `DeviceProtection`, `TechSupport`, `StreamingTV` y `StreamingMovies`, y
  corresponde a clientes con `InternetService = No`.

Estas categorías no son valores faltantes: indican que el cliente no tiene el
servicio base. Sin embargo, repiten información ya contenida en
`PhoneService` e `InternetService`. Esta redundancia se analiza en la
sección 4.

**Clasificación de las variables según lo que representan:**
- Identificador (no se usará como predictor): `customerID`.
- Variable objetivo: `Churn`.
- Numéricas (3): `tenure`, `MonthlyCharges`, `TotalCharges`.
- Categóricas (16): `gender`, `SeniorCitizen`, `Partner`, `Dependents`,
  `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`,
  `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`,
  `StreamingMovies`, `Contract`, `PaperlessBilling`, `PaymentMethod`.